In [2]:
#Importing needed libraries and texts
import nltk
from nltk.corpus import gutenberg

import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout
import time

random.seed(123)
np.random.seed(123)
tf.random.set_seed(123)

nltk.download('gutenberg')

author = "Jane Austen"
file_ids = ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt']
titles = ["Emma", "Persuasion", "Sense and Sensibility"]

combined_text = ""

for file_id, title in zip(file_ids, titles):
    text = gutenberg.raw(file_id)
    combined_text += text
    print(f"Title: {title}")
    print(f"File ID: {file_id}")
    print(f"Total Characters: {len(text):,}")
    print(f"Unique Characters: {len(set(text))}")
    print("-" * 40)

print(f"Author: {author}")
print(f"Combined Total Characters: {len(combined_text):,}")
print(f"Combined Unique Characters: {len(set(combined_text))}")

[nltk_data] Downloading package gutenberg to /root/nltk_data...


Title: Emma
File ID: austen-emma.txt
Total Characters: 887,071
Unique Characters: 77
----------------------------------------
Title: Persuasion
File ID: austen-persuasion.txt
Total Characters: 466,292
Unique Characters: 78
----------------------------------------
Title: Sense and Sensibility
File ID: austen-sense.txt
Total Characters: 673,022
Unique Characters: 78
----------------------------------------
Author: Jane Austen
Combined Total Characters: 2,026,385
Combined Unique Characters: 82


[nltk_data]   Unzipping corpora/gutenberg.zip.


In [3]:
#Creating character-to-index and index-to-character mappings
chars = sorted(list(set(combined_text)))

char_to_index = {char: i for i, char in enumerate(chars)}
index_to_char = {i: char for i, char in enumerate(chars)}

print(f"Total unique characters mapped: {len(chars)}")
print(f"Sample char_to_index: {list(char_to_index.items())[:5]}")
print(f"Sample index_to_char: {list(index_to_char.items())[:5]}")

Total unique characters mapped: 82
Sample char_to_index: [('\n', 0), (' ', 1), ('!', 2), ('"', 3), ('&', 4)]
Sample index_to_char: [(0, '\n'), (1, ' '), (2, '!'), (3, '"'), (4, '&')]


In [4]:
#Preparing overlapping input sequences and target characters
seq_length = 40
input_sequences = []
target_chars = []

for i in range(0, len(combined_text) - seq_length):
    input_seq = combined_text[i:i + seq_length]
    target_char = combined_text[i + seq_length]
    input_sequences.append([char_to_index[char] for char in input_seq])
    target_chars.append(char_to_index[target_char])

print(f"Total number of training sequences: {len(input_sequences):,}")
print(f"Sample input sequence (length {seq_length}): {input_sequences[0]}")
print(f"Sample target character index: {target_chars[0]}")

Total number of training sequences: 2,026,345
Sample input sequence (length 40): [52, 30, 68, 68, 56, 1, 57, 80, 1, 35, 56, 69, 60, 1, 26, 76, 74, 75, 60, 69, 1, 13, 20, 13, 18, 53, 0, 0, 47, 40, 37, 46, 38, 30, 1, 34, 0, 0, 28, 33]
Sample target character index: 26


In [5]:
#Building a character-level text generation model using LSTM
num_unique_chars = len(chars)

X = np.array(input_sequences)
y = np.array(target_chars)

model = Sequential([
    Embedding(input_dim=num_unique_chars, output_dim=128, input_length=seq_length),
    LSTM(256),
    Dense(num_unique_chars, activation='softmax')
])

model.build((None, seq_length))
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 40, 128)        │        10,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 82)             │        21,074 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 425,810 (1.62 MB)

 Trainable params: 425,810 (1.62 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
#Training the LSTM model
start_time = time.time()

history = model.fit(X, y, batch_size=128, epochs=10)

end_time = time.time()
training_time = end_time - start_time

print(f"Total Training Time: {training_time:.2f} seconds ({training_time / 60:.2f} minutes)")

Epoch 1/10
15831/15831 ━━━━━━━━━━━━━━━━━━━━ 149s 9ms/step - accuracy: 0.5411 - loss: 1.5402
Epoch 2/10
15831/15831 ━━━━━━━━━━━━━━━━━━━━ 144s 9ms/step - accuracy: 0.6146 - loss: 1.2508
Epoch 3/10
15831/15831 ━━━━━━━━━━━━━━━━━━━━ 146s 9ms/step - accuracy: 0.6297 - loss: 1.1882
Epoch 4/10
15831/15831 ━━━━━━━━━━━━━━━━━━━━ 145s 9ms/step - accuracy: 0.6372 - loss: 1.1586
Epoch 5/10
15831/15831 ━━━━━━━━━━━━━━━━━━━━ 145s 9ms/step - accuracy: 0.6423 - loss: 1.1398
Epoch 6/10
15831/15831 ━━━━━━━━━━━━━━━━━━━━ 145s 9ms/step - accuracy: 0.6459 - loss: 1.1262
Epoch 7/10
15831/15831 ━━━━━━━━━━━━━━━━━━━━ 145s 9ms/step - accuracy: 0.6485 - loss: 1.1161
Epoch 8/10
15831/15831 ━━━━━━━━━━━━━━━━━━━━ 145s 9ms/step - accuracy: 0.6506 - loss: 1.1086
Epoch 9/10
15831/15831 ━━━━━━━━━━━━━━━━━━━━ 146s 9ms/step - accuracy: 0.6521 - loss: 1.1026
Epoch 10/10
15831/15831 ━━━━━━━━━━━━━━━━━━━━ 145s 9ms/step - accuracy: 0.6533 - loss: 1.0979
Total Training Time: 1514.71 seconds (25.25 minutes)


In [13]:
#Generating three text samples using the trained model and three different sampling temperatures
def generate_text(model, start_string, num_generate=300, temperature=1.0,
                  seq_length=40, enforce_length=True):
    if enforce_length:
        assert len(start_string) == seq_length, \
            f"Seed must be exactly {seq_length} chars, got {len(start_string)}"

    input_eval = [char_to_index[s] for s in start_string]
    text_generated = []

    for _ in range(num_generate):
        eval_tensor = tf.expand_dims(input_eval, 0)
        probs = model(eval_tensor, training=False)[0]

        logits = tf.math.log(probs + 1e-8) / temperature
        predicted_id = int(tf.random.categorical(logits[None, :], num_samples=1)[0, 0])

        text_generated.append(index_to_char[predicted_id])
        input_eval.append(predicted_id)
        input_eval = input_eval[1:]

    return start_string + ''.join(text_generated)


short_seed = "It is a truth universally"
print(f"Short seed length: {len(short_seed)}")

full_seed = "It is a truth universally acknowledged, "
print(f"Full seed length: {len(full_seed)}")

long_seed = "It is a truth universally acknowledged, that a single man in possession of a good fortune, "
print(f"Long seed length: {len(long_seed)}")

for temp in [0.2, 0.7, 1.2]:
    print(f"\n=== Temperature: {temp} ===")

    tf.random.set_seed(123)
    print(f"\n[25-char seed]\n{generate_text(model, short_seed, 300, temp, enforce_length=False)}")

    tf.random.set_seed(123)
    print(f"\n[40-char seed]\n{generate_text(model, full_seed, 300, temp)}")

    tf.random.set_seed(123)
    print(f"\n[91-char seed]\n{generate_text(model, long_seed, 300, temp, enforce_length=False)}")

Short seed length: 25
Full seed length: 40
Long seed length: 91

=== Temperature: 0.2 ===

[25-char seed]
It is a truth universally distressed at the party to his letters,
and had been all that she had no doubt of his constant to the party to his son,
and the subject any thing in the same time as to her sister's complaint and allowed to be a little attachment to the same time,
and all the complaints of his father's being
in the

[40-char seed]
It is a truth universally acknowledged, and they were gone by the party which had been a great deal of more than the same time, and as he was so soon afterwards,
and was to be safely to be as much to be a sound of his father's constant and agitation of his sister's account
of the same time to her at all to his father's account,
and as she

[91-char seed]
It is a truth universally acknowledged, that a single man in possession of a good fortune, and the party was so little assistance to be standing of the party to be safely attentions
of the compla